## Actividad 1: Carga con dtypes explícitos




### patients.csv


#### Carga de datos "ingenua"

In [1]:
import pandas as pd

patients_naive = pd.read_csv("C:/Users/Propietarioi/scripts/DataScience/output/csv/patients.csv")
patients_memory_naive=patients_naive.memory_usage(deep=True).sum() / 1e6
patients_naive.info(memory_usage="deep") 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22959 entries, 0 to 22958
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   22959 non-null  object 
 1   BIRTHDATE            22959 non-null  object 
 2   DEATHDATE            2959 non-null   object 
 3   SSN                  22959 non-null  object 
 4   DRIVERS              19100 non-null  object 
 5   PASSPORT             17977 non-null  object 
 6   PREFIX               18522 non-null  object 
 7   FIRST                22959 non-null  object 
 8   MIDDLE               18392 non-null  object 
 9   LAST                 22959 non-null  object 
 10  SUFFIX               253 non-null    object 
 11  MAIDEN               6293 non-null   object 
 12  MARITAL              15566 non-null  object 
 13  RACE                 22959 non-null  object 
 14  ETHNICITY            22959 non-null  object 
 15  GENDER               22959 non-null 

Tiene sentido cargar las 28 columnas?

In [2]:
#Revisamos cuántos valores únicos tiene cada una
for col in ["GENDER", "RACE", "ETHNICITY", "CITY", "STATE"]:
    print(col, patients_naive[col].nunique())

GENDER 2
RACE 6
ETHNICITY 2
CITY 421
STATE 1


**STATE** es un valor único, no nos brinda información. **CITY** sí nos permite agrupar pacientes, sin embargo no tenemos una pregunta para la cual sea relevante conservar esta columna.
**GENDER, RACE Y ETHNICITY** son valores únicos manejables a comparación de las 22,959 filas que repiten esos strings, candidatos a "category".

Carga de datos optimizada

In [3]:
#Filter cols
dtypes_patients= {
    "GENDER":"category",
    "RACE":"category",
    "ETHNICITY":"category",
}

parse_dates_patients = ["BIRTHDATE", "DEATHDATE"]
columns_to_use= ["Id", "BIRTHDATE", "DEATHDATE", "GENDER", "RACE", "ETHNICITY"]


In [4]:
patients=pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/patients.csv",
    usecols= columns_to_use,
    dtype=dtypes_patients,
    parse_dates=parse_dates_patients
)
memory_patients=patients.memory_usage(deep=True).sum()/1e6

print(f"Before: {patients_memory_naive:.1f} MB | After: {memory_patients:.1f} MB")

#del patients_naive

Before: 31.4 MB | After: 2.6 MB


**Identificadores**: `Id`. 
**Variables categóricas**: `GENDER`, `RACE`, `ETHNICITY`.
**Fechas**: `BIRTHDATE`, `DEATHDATE`. 

**Resultado**: Se redujo el uso de memoria de 31.4 MB a 2.6 MB, una reducción de 91.7%. La mayor parte del ahorro viene de decidir usar solo 6 columnas de 28 en el DataFrame. Excluimos `STATE` (un valor único, que no aporta información) y las columnas de identidad que no se utilizan en el
análisis (SSN, nombres, dirección, etc.).


--- 
### encounters.csv


#### Carga de datos "ingenua"

In [5]:
#Naive loading (just to compare memory usage)

encounters_naive=pd.read_csv("C:/Users/Propietarioi/scripts/DataScience/output/csv/encounters.csv")
memory_naive_encounters=encounters_naive.memory_usage(deep=True).sum()/1e6

In [6]:
encounters_full_check = pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/encounters.csv",
    usecols=["Id", "START", "STOP", "PATIENT", "ENCOUNTERCLASS", "CODE", "DESCRIPTION"]
)
#quiero verificar las columnas que quiero usar para category.
for col in ["ENCOUNTERCLASS", "CODE","DESCRIPTION"]:
    print(col, encounters_full_check[col].nunique())

ENCOUNTERCLASS 10
CODE 63
DESCRIPTION 63


#### Carga de datos optimizada

In [7]:
#optimized loading (again, just to compare)
columns_to_encounters= ["Id","START","STOP","PATIENT", "ENCOUNTERCLASS","CODE", "DESCRIPTION"]

dtypes_encounters= {
    "ENCOUNTERCLASS": "category",
    "CODE": "category",
    "DESCRIPTION":"category",
}

parse_dates_encounters= ["START","STOP"]

encounters=pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/encounters.csv",
    usecols=columns_to_encounters,
    dtype=dtypes_encounters,
    parse_dates=parse_dates_encounters
)
memory_encounters=encounters.memory_usage(deep=True).sum()/1e6

In [8]:
encounters.info(memory_usage="deep")
encounters.memory_usage(deep=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1367553 entries, 0 to 1367552
Data columns (total 7 columns):
 #   Column          Non-Null Count    Dtype              
---  ------          --------------    -----              
 0   Id              1367553 non-null  object             
 1   START           1367553 non-null  datetime64[ns, UTC]
 2   STOP            1367553 non-null  datetime64[ns, UTC]
 3   PATIENT         1367553 non-null  object             
 4   ENCOUNTERCLASS  1367553 non-null  category           
 5   CODE            1367553 non-null  category           
 6   DESCRIPTION     1367553 non-null  category           
dtypes: category(3), datetime64[ns, UTC](2), object(2)
memory usage: 267.4 MB


Index                   128
Id                127182429
START              10940424
STOP               10940424
PATIENT           127182429
ENCOUNTERCLASS      1368500
CODE                1373814
DESCRIPTION         1375871
dtype: int64

In [9]:
print(f"Before: {memory_naive_encounters:.1f} MB | After: {memory_encounters: .1f} MB")

Before: 1213.9 MB | After:  280.4 MB



**Identificadores**: `Id`/`PATIENT`. **Variables categóricas**: `ENCOUNTERCLASS`/`CODE`/`DESCRIPTION`
**Fechas**: `START`/`STOP` 

Se omiten las columnas de costos y de organización/proveedor, ya que no se utilizan en
este análisis.

**Resultado:** Se redujeron 1213,9 MB a 280,4 MB (reducción del 77 %). El espacio realmente ahorrado se debió
casi en su totalidad a que se evitó la re-inferencia de tipos costosos; los identificadores (`Id`,
`PATIENT`) siguen representando más del 95 % del uso de memoria porque son UUID únicos
por fila `category` no ayuda en este caso, por lo que no se optimizan más.

--- 
###  observations.csv


#### Estimación de carga de datos "ingenua"

Este archivo parece ser aún más pesado que los anteriores entonces se hará una estimación del tamaño del archivo original a partir de una muestra

In [10]:
obs_sample=pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/observations.csv",
    nrows=5
)
obs_sample

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
0,2024-01-01T17:19:31Z,c0eb3b06-a197-c4c0-d34b-2cfdd1006a79,c0eb3b06-a197-c4c0-ea38-2ed7261f612d,vital-signs,8302-2,Body Height,54.4,cm,numeric
1,1997-01-20T01:00:13Z,a79fbcb1-7867-a5fd-484c-16f0dbb2d8ef,a79fbcb1-7867-a5fd-5372-1d4e5580b7d9,vital-signs,8302-2,Body Height,49.5,cm,numeric
2,2026-02-01T01:38:08Z,c4b895fc-1b4d-b904-fbab-55ca67595243,c4b895fc-1b4d-b904-6766-c912dff76eb4,vital-signs,8302-2,Body Height,57.9,cm,numeric
3,2024-01-01T17:19:31Z,c0eb3b06-a197-c4c0-d34b-2cfdd1006a79,c0eb3b06-a197-c4c0-ea38-2ed7261f612d,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,0.0,{score},numeric
4,1997-01-20T01:00:13Z,a79fbcb1-7867-a5fd-484c-16f0dbb2d8ef,a79fbcb1-7867-a5fd-5372-1d4e5580b7d9,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,1.0,{score},numeric


In [11]:
obs_sample_check=pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/observations.csv",
        nrows=100000
)

obs_sample_check.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   DATE         100000 non-null  object
 1   PATIENT      100000 non-null  object
 2   ENCOUNTER    94219 non-null   object
 3   CATEGORY     94219 non-null   object
 4   CODE         100000 non-null  object
 5   DESCRIPTION  100000 non-null  object
 6   VALUE        100000 non-null  object
 7   UNITS        68782 non-null   object
 8   TYPE         100000 non-null  object
dtypes: object(9)
memory usage: 63.5 MB


In [12]:
#to estimate the actual size
with open ("C:/Users/Propietarioi/scripts/DataScience/output/csv/observations.csv", encoding="utf-8") as f:
    total_rows= sum(1 for _ in f)-1 #-1 condidering header
print(total_rows)

17699379


In [13]:
factor= total_rows / 100000
estimated_mb=63.5*factor
print(f"estimated memory usage (naive, complete):{estimated_mb: .0f} MB (~{estimated_mb/1000: .1f} GB)")

estimated memory usage (naive, complete): 11239 MB (~ 11.2 GB)


Comprobamos que la memoria que usará este archivo es considerablemente mayor a las anteriores (~ 11 GB), por lo que es prioridad la carga optimiazda.

#### Carga de datos "ingenua" 

In [ ]:
observations_naive= pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/observations.csv"
)

memory_naive_observations= observations_naive.memory_usage(deep=True).sum()/1e6

print(f"Before: {memory_naive_observations: .1f} MB")

Before:  11889.7 MB


In [15]:
#Turn to a number, anything that can´t be converted becomes a NaN

value_numeric_test = pd.to_numeric(observations_naive["VALUE"], errors="coerce")
pct_non_numeric = value_numeric_test.isna().mean() * 100
print(f"% of VALUE that is not convertible to a number: {pct_non_numeric: .2f}%")

#examples of non numeric values
non_numeric= observations_naive.loc[value_numeric_test.isna(), "VALUE"].unique()
print(non_numeric[:20])

% of VALUE that is not convertible to a number:  36.88%
['Never smoked tobacco (finding)' 'Sudden cardiac death (disorder)' 'No'
 'Yes' 'A little bit' '5 or more times a week' 'Food' 'None/uninsured'
 'Somewhat' 'Less than once a week' 'I choose not to answer this question'
 'Medicare' 'Full-time work' 'More than high school' '981 Kuhlman Vista'
 'I have housing' 'Language other than English' 'Pacific Islander'
 'Unemployed' 'High school diploma or GED']


 **`VALUE` es una columna mixta.**
 
Se intentó convertir VALUE a numérico con pd.to_numeric(errors="coerce"),
que convierte a NaN cualquier valor no numérico en vez de fallar.

Resultado: 36.88% de las filas no son convertibles a número.

Al inspeccionar ejemplos de esos valores no numéricos, se observó que
corresponden a respuestas de cuestionarios sociales/de estilo de vida
(tabaquismo, seguro médico, empleo, vivienda), es decir, no son errores de datos,
sino un tipo de observación completamente distinto a las mediciones
clínicas numéricas (altura, presión, laboratorios).

Posteriormente se separará la columna de `VALUE` en 2 columnas, lo que es convertible y lo que no. 


In [16]:
observations_naive.groupby("TYPE")["VALUE"].apply(lambda x:pd.to_numeric(x,errors="coerce").notna().mean())

TYPE
numeric    1.000000
text       0.038015
Name: VALUE, dtype: float64

In [17]:
for col in ["CATEGORY", "CODE", "DESCRIPTION", "UNITS", "TYPE"]:
    print(col, observations_naive[col].nunique())

CATEGORY 8
CODE 299
DESCRIPTION 301
UNITS 51
TYPE 2


In [18]:
columns_to_observations=["DATE","PATIENT","ENCOUNTER","CATEGORY", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"]
dtypes_observations = {
    "CATEGORY": "category",
    "CODE": "category",
    "DESCRIPTION": "category",
    "UNITS": "category",
    "TYPE": "category",
    "PATIENT": "category",
    "ENCOUNTER": "category",
}
parse_dates_observations= ["DATE"]

observations=pd.read_csv(
    "C:/Users/Propietarioi/scripts/DataScience/output/csv/observations.csv",
    usecols=columns_to_observations,
    dtype=dtypes_observations,
    parse_dates=parse_dates_observations
)

In [19]:
#clean VALUE columns 
VALUE_TEXT= observations["VALUE"].where(observations["TYPE"]=="text")
VALUE_NUMERIC= pd.to_numeric(
     observations["VALUE"].where(observations["TYPE"]== "numeric"),
     errors="coerce"
)

observations["VALUE_TEXT"] = VALUE_TEXT.astype("category")
observations["VALUE_NUMERIC"] = VALUE_NUMERIC.astype("float32")

observations = observations.drop(columns=["VALUE"])
observations

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,UNITS,TYPE,VALUE_TEXT,VALUE_NUMERIC
0,2024-01-01 17:19:31+00:00,c0eb3b06-a197-c4c0-d34b-2cfdd1006a79,c0eb3b06-a197-c4c0-ea38-2ed7261f612d,vital-signs,8302-2,Body Height,cm,numeric,NaN,54.400002
1,1997-01-20 01:00:13+00:00,a79fbcb1-7867-a5fd-484c-16f0dbb2d8ef,a79fbcb1-7867-a5fd-5372-1d4e5580b7d9,vital-signs,8302-2,Body Height,cm,numeric,NaN,49.500000
2,2026-02-01 01:38:08+00:00,c4b895fc-1b4d-b904-fbab-55ca67595243,c4b895fc-1b4d-b904-6766-c912dff76eb4,vital-signs,8302-2,Body Height,cm,numeric,NaN,57.900002
3,2024-01-01 17:19:31+00:00,c0eb3b06-a197-c4c0-d34b-2cfdd1006a79,c0eb3b06-a197-c4c0-ea38-2ed7261f612d,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,{score},numeric,NaN,0.000000
4,1997-01-20 01:00:13+00:00,a79fbcb1-7867-a5fd-484c-16f0dbb2d8ef,a79fbcb1-7867-a5fd-5372-1d4e5580b7d9,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,{score},numeric,NaN,1.000000
...,...,...,...,...,...,...,...,...,...,...
17699374,2022-06-10 19:33:16+00:00,0bfad66b-de8e-5546-5db4-a0334e2a67c2,NaN,NaN,QOLS,QOLS,{score},numeric,NaN,0.700000
17699375,2023-06-10 20:33:16+00:00,0bfad66b-de8e-5546-5db4-a0334e2a67c2,NaN,NaN,QOLS,QOLS,{score},numeric,NaN,0.700000
17699376,2024-06-10 20:33:16+00:00,0bfad66b-de8e-5546-5db4-a0334e2a67c2,NaN,NaN,QOLS,QOLS,{score},numeric,NaN,0.700000
17699377,2025-06-10 20:33:16+00:00,0bfad66b-de8e-5546-5db4-a0334e2a67c2,NaN,NaN,QOLS,QOLS,{score},numeric,NaN,0.700000


In [20]:
memory_observations = observations.memory_usage(deep=True).sum() / 1e6
reduction_pct = (1 - memory_observations / memory_naive_observations) * 100

print(f"Before: {memory_naive_observations:.1f} MB | After: {memory_observations:.1f} MB")
print(f"Reduction: {reduction_pct:.1f}%")

Before: 11889.7 MB | After: 605.6 MB
Reduction: 94.9%


In [21]:
observations_naive.memory_usage(deep=True)
observations.memory_usage(deep=True)

Index                  128
DATE             141595032
PATIENT           38062369
ENCOUNTER        156864178
CATEGORY          17700203
CODE              35426069
DESCRIPTION       35438912
UNITS             17704641
TYPE              17699612
VALUE_TEXT        74272321
VALUE_NUMERIC     70797516
dtype: int64

In [22]:
observations.memory_usage(deep=True)

Index                  128
DATE             141595032
PATIENT           38062369
ENCOUNTER        156864178
CATEGORY          17700203
CODE              35426069
DESCRIPTION       35438912
UNITS             17704641
TYPE              17699612
VALUE_TEXT        74272321
VALUE_NUMERIC     70797516
dtype: int64

In [23]:
mem_antes = observations_naive.memory_usage(deep=True)
mem_despues = observations.memory_usage(deep=True)

resumen = pd.DataFrame({
    "dtype antes": observations_naive.dtypes.astype(str),
    "MB antes": (mem_antes / 1e6).round(1),
    "dtype después": observations.dtypes.reindex(observations_naive.columns).astype(str),
    "MB después": (mem_despues / 1e6).reindex(observations_naive.columns).round(1),
})

# Quitar la fila "Index" (no es una columna de datos)
resumen = resumen.drop(index="Index")

# Agregar manualmente las filas de VALUE_TEXT y VALUE_NUMERIC,
# y quitar la fila VALUE original (ya no aplica tal cual)
resumen = resumen.drop(index="VALUE")

resumen.loc["VALUE_TEXT"] = [
    "object (parte de VALUE)", 1166.7,  # MB antes: referencia a la columna original completa
    "category", (mem_despues["VALUE_TEXT"] / 1e6).round(1)
]
resumen.loc["VALUE_NUMERIC"] = [
    "object (parte de VALUE)", None,  # ya contado arriba, evitamos doble conteo
    "float32", (mem_despues["VALUE_NUMERIC"] / 1e6).round(1)
]

resumen

C:\Users\Propietarioi\AppData\Local\Temp\ipykernel_25624\765117472.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  resumen.loc["VALUE_NUMERIC"] = [


,dtype antes,MB antes,dtype después,MB después
CATEGORY,object,1145.9,category,17.7
CODE,object,1122.9,category,35.4
DATE,object,1362.9,"datetime64[ns, UTC]",141.6
DESCRIPTION,object,1767.5,category,35.4
ENCOUNTER,object,1607.8,category,156.9
PATIENT,object,1646.0,category,38.1
TYPE,object,1112.4,category,17.7
UNITS,object,957.5,category,17.7
VALUE_TEXT,object (parte de VALUE),1166.7,category,74.3
VALUE_NUMERIC,object (parte de VALUE),NaN,float32,70.8


In [24]:
print(resumen.to_markdown())

|               | dtype antes             |   MB antes | dtype después       |   MB después |
|:--------------|:------------------------|-----------:|:--------------------|-------------:|
| CATEGORY      | object                  |     1145.9 | category            |         17.7 |
| CODE          | object                  |     1122.9 | category            |         35.4 |
| DATE          | object                  |     1362.9 | datetime64[ns, UTC] |        141.6 |
| DESCRIPTION   | object                  |     1767.5 | category            |         35.4 |
| ENCOUNTER     | object                  |     1607.8 | category            |        156.9 |
| PATIENT       | object                  |     1646   | category            |         38.1 |
| TYPE          | object                  |     1112.4 | category            |         17.7 |
| UNITS         | object                  |      957.5 | category            |         17.7 |
| VALUE_TEXT    | object (parte de VALUE) |     1166.7 | cat

## Actividad 3

In [25]:
patients.isna().mean ()*100

Id            0.000000
BIRTHDATE     0.000000
DEATHDATE    87.111808
RACE          0.000000
ETHNICITY     0.000000
GENDER        0.000000
dtype: float64

In [26]:
encounters.isna().mean() * 100

Id                0.0
START             0.0
STOP              0.0
PATIENT           0.0
ENCOUNTERCLASS    0.0
CODE              0.0
DESCRIPTION       0.0
dtype: float64

In [27]:
observations.isna().mean() * 100

DATE              0.000000
PATIENT           0.000000
ENCOUNTER         3.537938
CATEGORY          3.537938
CODE              0.000000
DESCRIPTION       0.000000
UNITS            27.157377
TYPE              0.000000
VALUE_TEXT       61.660649
VALUE_NUMERIC    38.339351
dtype: float64

In [28]:
sin_encuentro=observations[observations["ENCOUNTER"].isna()]
sin_encuentro["CATEGORY"].value_counts()

CATEGORY
exam              0
imaging           0
laboratory        0
procedure         0
social-history    0
survey            0
therapy           0
vital-signs       0
Name: count, dtype: int64

In [29]:
sin_encuentro["CATEGORY"].value_counts().head(10)

CATEGORY
exam              0
imaging           0
laboratory        0
procedure         0
social-history    0
survey            0
therapy           0
vital-signs       0
Name: count, dtype: int64

In [30]:
sin_encounter = observations[observations["ENCOUNTER"].isna()]
sin_encounter["DESCRIPTION"].value_counts().head(10)

DESCRIPTION
DALY                                                                                  208731
QALY                                                                                  208731
QOLS                                                                                  208731
Alanine aminotransferase [Enzymatic activity/volume] in Serum or Plasma                    0
Albumin [Mass/volume] in Serum or Plasma                                                   0
Alkaline phosphatase [Enzymatic activity/volume] in Serum or Plasma                        0
American house dust mite IgE Ab [Units/volume] in Serum                                    0
Appearance of Urine                                                                        0
Are you a refugee                                                                          0
Are you covered by health insurance or some other kind of health care plan [PhenX]         0
Name: count, dtype: int64

In [31]:
observations[observations["UNITS"].isna()]["TYPE"].value_counts()

TYPE
text       4806421
numeric        266
Name: count, dtype: int64

In [32]:
numeric_sin_units= observations[(observations["UNITS"].isna()) & (observations["TYPE"]=="numeric")]
numeric_sin_units["DESCRIPTION"].value_counts()

DESCRIPTION
Position of body and posture (observable entity)                       90
Cobb angle (observable entity)                                         88
Risser sign (finding)                                                  88
Alkaline phosphatase [Enzymatic activity/volume] in Serum or Plasma     0
American house dust mite IgE Ab [Units/volume] in Serum                 0
                                                                       ..
Body mass index (BMI) [Percentile] Per age and sex                      0
Body mass index (BMI) [Ratio]                                           0
Body temperature                                                        0
C reactive protein [Mass/volume] in Serum or Plasma                     0
Address                                                                 0
Name: count, Length: 301, dtype: int64